# Exercise 09 : Offload and Measure

- A large tool result can fill the next model call ; offloading replaces it with a path and a preview
- The QuickJS interpreter can compute over rows that never enter the prompt
- Input tokens are the figure to watch when you change how evidence reaches the model

Optional. Measure input tokens for the same Aurelia manuals export with and without
offloading, then total ticket hours in QuickJS. Report the numbers in a table.
All data is made up.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [ ]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

## Step 1 : Export without offloading

In [ ]:
import json
import sqlite3
import threading
import time
from pathlib import Path

from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import AIMessage, HumanMessage
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from deepagents import create_deep_agent
from deepagents.backends import StateBackend, FilesystemBackend, CompositeBackend
from deepagents.backends.utils import create_file_data
from langgraph.checkpoint.memory import InMemorySaver


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.reset()

    def reset(self):
        self.calls = self.input = self.output = self.invalid = self.errors = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for generations in response.generations:
                for generation in generations:
                    message = getattr(generation, "message", None)
                    usage = getattr(message, "usage_metadata", None) or {}
                    self.input += usage.get("input_tokens", 0)
                    self.output += usage.get("output_tokens", 0)
                    self.invalid += len(getattr(message, "invalid_tool_calls", []) or [])

    def on_llm_error(self, error, **kwargs):
        with self.lock:
            self.errors += 1


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=1800, max_retries=0)
LIMIT = {"recursion_limit": 36}
DB = Path(data("capstone/service.db")).resolve()
MANUALS = Path(data_folder("capstone/manuals")).resolve()
PAGES = {p.name: p.read_text(encoding="utf-8") for p in sorted(MANUALS.glob("*.md"))}


def ask(text):
    return {"messages": [HumanMessage(text)]}


def answer(result):
    return str(next(m.content for m in reversed(result["messages"])
                    if isinstance(m, AIMessage) and m.content and not m.tool_calls))


def measured(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return operation()
    finally:
        row = dict(label=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, malformed=counter.invalid, errors=counter.errors,
                   seconds=round(time.perf_counter() - start, 1))
        print("metrics :", json.dumps(row))
        measured.last = row


async def measured_async(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return await operation()
    finally:
        row = dict(label=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, malformed=counter.invalid, errors=counter.errors,
                   seconds=round(time.perf_counter() - start, 1))
        print("metrics :", json.dumps(row))
        measured.last = row


def show(result):
    print("tools :", [m.name for m in result["messages"] if m.type == "tool"])
    print("answer :", answer(result))


@tool
def part_stock(part: str) -> str:
    """Look up stock, depot and name for one Aurelia part number."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, name, stock, depot FROM parts WHERE part_no = ?",
                          (part.upper(),)).fetchall()
    return json.dumps(rows)

from deepagents.middleware.filesystem import FilesystemMiddleware
from langchain.agents.middleware import wrap_model_call


@tool
def all_manuals() -> str:
    """Export all Aurelia manuals with their source names."""
    return "\n\n".join(f"[{name}]\n{text}" for name, text in PAGES.items())


raw_chars = len(all_manuals.invoke({}))
print("raw characters :", raw_chars)

plain = create_deep_agent(model=llm, tools=[all_manuals], backend=StateBackend(),
    system_prompt="Call all_manuals once. Answer with the approximate character count only.")
result = measured("no-offload", lambda: plain.invoke(ask(
    "Call all_manuals once. How large is the export, roughly? Do not quote manuals."), LIMIT))
show(result)
no_offload = measured.last

In [ ]:
print("check :", "PASS" if no_offload["input_tokens"] > 0 and raw_chars > 10_000 else "FAIL")
assert no_offload["input_tokens"] > 0 and raw_chars > 10_000

## Step 2 : Export with offloading

Lower `tool_token_limit_before_evict` so the export is saved to the filesystem
and replaced with a path and preview. The limit is in tokens, counted as
characters / 4 : a result of at least `4 * limit` characters is evicted.

The `FilesystemMiddleware` you pass replaces the one `create_deep_agent` builds,
and the harness does not copy `permissions=` onto a replacement (deepagents
0.7.30). The cell passes `_permissions=deny_reread` to the override as well.
The system prompt differs from step 1, so part of the token change is the prompt.

In [ ]:
from deepagents import FilesystemPermission

captured = []


@wrap_model_call
def capture_tool_results(request, handler):
    captured.extend(str(m.content) for m in request.messages if m.type == "tool")
    return handler(request)


backend = StateBackend()
# Deny re-reading the evicted export. A FilesystemMiddleware passed in `middleware`
# replaces the harness's own, configuration included : it needs the rules itself.
deny_reread = [
    FilesystemPermission(operations=["read"], paths=["/large_tool_results/**"], mode="deny"),
]
offloader = create_deep_agent(
    model=llm,
    tools=[all_manuals],
    backend=backend,
    permissions=deny_reread,
    middleware=[
        FilesystemMiddleware(backend=backend, tool_token_limit_before_evict=____,
                             _permissions=deny_reread),
        capture_tool_results,
    ],
    system_prompt=(
        "Call all_manuals once. Report the saved result path from the tool preview. "
        "Do not call read_file. Do not use the shell. Then stop."
    ),
)
result = measured("offload", lambda: offloader.invoke(ask(
    "Call all_manuals once. Report only the saved path from the tool preview, then stop."), LIMIT))
show(result)
print("model tool results :", [text[:240] for text in captured])
print("tool message chars :", [len(text) for text in captured])
print("saved paths :", sorted(result.get("files", {})))
offload = measured.last
offload_tool_chars = max((len(text) for text in captured), default=0)

<details><summary>Hint</summary>

`600`, as in demo03.

</details>

In [ ]:
previewed = any("/large_tool_results/" in text for text in captured) or bool(result.get("files"))
shorter_tool = offload_tool_chars < raw_chars // 2
fewer_tokens = offload["input_tokens"] < no_offload["input_tokens"]
print("check :", "PASS" if previewed and shorter_tool else "FAIL",
      {"no_offload_tokens": no_offload["input_tokens"], "offload_tokens": offload["input_tokens"],
       "offload_tool_chars": offload_tool_chars, "raw_chars": raw_chars,
       "fewer_tokens": fewer_tokens})
assert previewed and shorter_tool

The next cell needs no model. A scripted model calls `read_file` on an evicted
path, once with the override missing its rules and once with them, so you can
see the deny fire.

In [ ]:
from deepagents.backends.utils import create_file_data
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel


class Scripted(GenericFakeChatModel):
    def bind_tools(self, tools, **kwargs):
        return self


def reread(middleware):
    script = iter([AIMessage("", tool_calls=[{"name": "read_file", "id": "r1",
                                              "args": {"file_path": "/large_tool_results/x"}}]),
                   AIMessage("done")])
    probe = create_deep_agent(model=Scripted(messages=script), backend=StateBackend(),
                              permissions=deny_reread, middleware=[middleware])
    out = probe.invoke({"messages": [HumanMessage("go")],
                        "files": {"/large_tool_results/x": create_file_data("FULL EXPORT")}})
    return next(str(m.content) for m in out["messages"] if m.type == "tool")


without_rules = reread(FilesystemMiddleware(backend=StateBackend(), tool_token_limit_before_evict=600))
with_rules = reread(FilesystemMiddleware(backend=StateBackend(), tool_token_limit_before_evict=600,
                                         _permissions=deny_reread))
print("override without _permissions :", without_rules)
print("override with _permissions    :", with_rules)
print("check :", "PASS" if "permission denied" in with_rules else "FAIL")
assert "permission denied" in with_rules

## Step 3 : Tickets in QuickJS

Load `data/capstone/tickets.csv` through a PTC tool and compute inside `eval`.

In [ ]:
import csv
import importlib.metadata as metadata
import xml.etree.ElementTree as ET
from decimal import Decimal
from langchain_quickjs import CodeInterpreterMiddleware, PTCOption

with Path(data("capstone/tickets.csv")).open(encoding="utf-8", newline="") as handle:
    tickets = list(csv.DictReader(handle))


@tool
def ticket_rows() -> str:
    """Load all Aurelia tickets as JSON rows with site_id, hours and parts_used."""
    return json.dumps(tickets)


ptc: PTCOption = [ticket_rows]
interpreter = CodeInterpreterMiddleware(
    ptc=ptc, subagents=False, mode="turn",
    memory_limit=16 * 1024 * 1024, timeout=2, max_ptc_calls=2, max_result_chars=1200)
analyst = create_deep_agent(model=llm, middleware=[interpreter], system_prompt=(
    "Use only eval. Load ticket data inside JavaScript with "
    "JSON.parse(await tools.ticketRows({})). Compute from those rows; do not return "
    "or print the raw rows. Return the requested JSON using JSON.stringify as the "
    "last expression. Then answer briefly from the result."))
print("versions :", metadata.version("deepagents"), metadata.version("langchain-quickjs"))
result = await measured_async("quickjs-tickets", lambda: analyst.ainvoke(ask(
    "For site S01, count tickets and sum hours only where parts_used is empty. "
    "Return JSON with count and hours."), LIMIT))
computed = None
for message in result["messages"]:
    for call in getattr(message, "tool_calls", []) or []:
        if call["name"] == "eval":
            print("JavaScript :\n" + call["args"]["code"])
    if message.type == "tool" and message.name == "eval":
        print("sandbox :", message.content)
        computed = json.loads(ET.fromstring(message.content).text)
show(result)
selected = [row for row in tickets if row["site_id"] == "S01" and not row["parts_used"]]
expected = {"count": len(selected), "hours": float(sum(Decimal(row["hours"]) for row in selected))}
print("Python check :", expected)
print("sandbox :", computed)
quickjs = measured.last
____

<details><summary>Hint</summary>

Assert that the sandbox `count` and `hours` match the Python `expected` dict.

</details>

In [ ]:
print("check :", "PASS" if computed == expected or
      {k: computed[k] for k in expected} == expected else "FAIL")
assert {key: computed[key] for key in expected} == expected

## Step 4 : The table

In [ ]:
rows = [
    ("no offload", no_offload["input_tokens"], no_offload["calls"], no_offload["seconds"], raw_chars),
    ("offload", offload["input_tokens"], offload["calls"], offload["seconds"], offload_tool_chars),
    ("quickjs tickets", quickjs["input_tokens"], quickjs["calls"], quickjs["seconds"], "n/a"),
]
print("| path | input tokens | calls | seconds | tool chars seen |")
print("| --- | --- | --- | --- | --- |")
for label, tokens, calls, seconds, chars in rows:
    print(f"| {label} | {tokens} | {calls} | {seconds} | {chars} |")
print("sandbox totals :", computed)
print("raw export chars :", raw_chars)
____

<details><summary>Hint</summary>

Assert `offload_tool_chars < raw_chars // 2` and that QuickJS recorded tokens.

</details>

## If time permits

- Raise the eviction limit until offloading stops, and find the threshold on this export
- Ask QuickJS for a breakdown by `error_code` at site `S03`
- Offload `ticket_rows` itself and compare with the interpreter path